In [0]:
def run_data_quality_checks(table, rd_df, dq_rules, run_id): ## table = target table. rd_df = retrieved data dataframe. dq_rules = data quality rules
    critical_failures = []
    warning_failures  = []
    dq_results = []

    table_rules = [r for r in dq_rules if r['table_name'] == table]

    logger.info(f"Data Quality Checks — analytics_lakehouse.{TARGET_SCHEMA}.{table}")

    for rule in table_rules:
        try:
            if rule['check_type'] == 'WHERE':
                fail_count = rd_df.where(rule['sql_expression']).count()
            elif rule['check_type'] == 'UNIQUE':
                fail_count = rd_df.groupBy(rule['sql_expression']).count().filter(F.col('count') > 1).count()
            else:
                raise Exception(f"Unsupported check type: {rule['check_type']}")

            if fail_count > 0 and rule['severity'] == 'CRITICAL':
                result = "FAIL"
                message = f"{fail_count} records failed"
                critical_failures.append({'table_name': rule['table_name'], 'rule_name': rule['rule_name'], 'fail_count': fail_count})
                logger.error(f"CRITICAL {rule['rule_name']}. {message}")

            elif fail_count > 0 and rule['severity'] == 'WARNING':
                result = "WARN"
                message = f"{fail_count} records failed"
                warning_failures.append({'table_name': rule['table_name'], 'rule_name': rule['rule_name'], 'fail_count': fail_count})
                logger.warning(f"WARNING {rule['rule_name']}. {message}")

            else:
                result = "PASS"
                message = None
                fail_count = 0
                logger.info(f"PASSED {rule['rule_name']}")

        except Exception as e:
            result = "FAIL"
            message = str(e)
            fail_count = -1
            logger.error(f"DQ check failed to execute for {rule['rule_name']}: {e}")

        dq_results.append({
            "run_id": run_id,
            "table_name": table,
            "rule_name": rule['rule_name'],
            "check_type": rule['check_type'],
            "sql_expression": rule['sql_expression'],
            "severity": rule['severity'],
            "result": result,
            "fail_count": fail_count,
            "message": message,
        })

    total = len(table_rules)
    passed = total - len(critical_failures) - len(warning_failures)
    logger.info(f"{total} checks | {passed} passed | {len(warning_failures)} warnings | {len(critical_failures)} critical")

    if critical_failures:
        failure_details = ', '.join([f"{f['rule_name']} ({f['fail_count']} rows)" for f in critical_failures])
        raise Exception(f"Data quality validation FAILED for {table}. Critical failures: {failure_details}")

    return dq_results